# Load Azure Databricks Audit Log Actions

This notebook creates a lookup table of Databricks audit log action_names with descriptions. Use Vector Search to find relevant actions based on user intent.

**Source:** https://learn.microsoft.com/en-us/azure/databricks/admin/system-tables/audit-logs

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
TABLE_NAME = "databricks_audit_actions"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

FRESH_START = True

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
# Schema: action_name is the primary key for Vector Search lookups
schema = StructType([
    StructField("action_name", StringType(), False),  # Primary key - what gets returned from VSI
    StructField("service_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("description", StringType(), True),
    StructField("embedding_text", StringType(), True),  # Rich text for semantic search
])

In [ ]:
# Comprehensive list of Databricks audit log actions with rich descriptions
# Format: (action_name, service_name, category, description)

AUDIT_ACTIONS = [
    # Unity Catalog - Table Operations
    ("createTable", "unityCatalog", "Table Operations", "Create a new table in Unity Catalog. Use this to find when tables were created, who created them, and what schema they have."),
    ("getTable", "unityCatalog", "Table Operations", "Access or retrieve table metadata from Unity Catalog. Use this to audit who viewed or accessed table information."),
    ("deleteTable", "unityCatalog", "Table Operations", "Delete a table from Unity Catalog. Use this to track table deletions and who removed data."),
    ("updateTable", "unityCatalog", "Table Operations", "Update table properties or metadata. Use this to track changes to table configurations."),
    ("listTables", "unityCatalog", "Table Operations", "List all tables in a schema. Use this to audit discovery and browsing of tables."),
    
    # Unity Catalog - Schema Operations
    ("createSchema", "unityCatalog", "Schema Operations", "Create a new schema (database) in Unity Catalog. Use this to track namespace creation."),
    ("deleteSchema", "unityCatalog", "Schema Operations", "Delete a schema from Unity Catalog. Use this to track schema removals."),
    ("updateSchema", "unityCatalog", "Schema Operations", "Update schema properties. Use this to track changes to schema configurations."),
    ("getSchema", "unityCatalog", "Schema Operations", "Get schema metadata. Use this to audit schema access."),
    ("listSchemas", "unityCatalog", "Schema Operations", "List all schemas in a catalog. Use this to audit schema discovery."),
    
    # Unity Catalog - Catalog Operations
    ("createCatalog", "unityCatalog", "Catalog Operations", "Create a new catalog in Unity Catalog. Use this to track top-level namespace creation."),
    ("deleteCatalog", "unityCatalog", "Catalog Operations", "Delete a catalog from Unity Catalog. Use this to track catalog removals."),
    ("updateCatalog", "unityCatalog", "Catalog Operations", "Update catalog properties. Use this to track changes to catalog configurations."),
    ("getCatalog", "unityCatalog", "Catalog Operations", "Get catalog metadata. Use this to audit catalog access."),
    ("listCatalogs", "unityCatalog", "Catalog Operations", "List all catalogs. Use this to audit catalog discovery."),
    
    # Unity Catalog - Volume Operations
    ("createVolume", "unityCatalog", "Volume Operations", "Create a new volume for file storage. Use this to track volume creation for unstructured data."),
    ("deleteVolume", "unityCatalog", "Volume Operations", "Delete a volume. Use this to track volume removals."),
    ("updateVolume", "unityCatalog", "Volume Operations", "Update volume properties. Use this to track changes to volume configurations."),
    ("getVolume", "unityCatalog", "Volume Operations", "Get volume metadata. Use this to audit volume access."),
    ("listVolumes", "unityCatalog", "Volume Operations", "List volumes in a schema. Use this to audit volume discovery."),
    
    # Unity Catalog - Function Operations
    ("createFunction", "unityCatalog", "Function Operations", "Create a user-defined function (UDF). Use this to track custom function deployments."),
    ("deleteFunction", "unityCatalog", "Function Operations", "Delete a user-defined function. Use this to track UDF removals."),
    ("updateFunction", "unityCatalog", "Function Operations", "Update function properties. Use this to track changes to UDF configurations."),
    ("getFunction", "unityCatalog", "Function Operations", "Get function metadata. Use this to audit function access."),
    ("listFunctions", "unityCatalog", "Function Operations", "List functions in a schema. Use this to audit function discovery."),
    
    # Unity Catalog - Permission Operations
    ("updatePermissions", "unityCatalog", "Permission Operations", "Modify permissions on a securable object like table, schema, or catalog. Use this to audit access control changes, grants, and revokes."),
    ("getPermissions", "unityCatalog", "Permission Operations", "Get permissions on a securable object. Use this to audit permission lookups."),
    ("getEffectivePermissions", "unityCatalog", "Permission Operations", "Get effective permissions for a principal. Use this to audit permission inheritance checks."),
    
    # Unity Catalog - External Locations
    ("createExternalLocation", "unityCatalog", "External Location Operations", "Create an external location pointing to cloud storage. Use this to track storage integration setup."),
    ("deleteExternalLocation", "unityCatalog", "External Location Operations", "Delete an external location. Use this to track storage integration removals."),
    ("updateExternalLocation", "unityCatalog", "External Location Operations", "Update external location properties. Use this to track storage configuration changes."),
    ("getExternalLocation", "unityCatalog", "External Location Operations", "Get external location metadata. Use this to audit storage integration access."),
    ("listExternalLocations", "unityCatalog", "External Location Operations", "List external locations. Use this to audit storage integration discovery."),
    
    # Unity Catalog - Storage Credentials
    ("createStorageCredential", "unityCatalog", "Storage Credential Operations", "Create a storage credential for cloud access. Use this to track credential creation for Azure, AWS, or GCP storage."),
    ("deleteStorageCredential", "unityCatalog", "Storage Credential Operations", "Delete a storage credential. Use this to track credential removals."),
    ("updateStorageCredential", "unityCatalog", "Storage Credential Operations", "Update storage credential properties. Use this to track credential configuration changes."),
    ("getStorageCredential", "unityCatalog", "Storage Credential Operations", "Get storage credential metadata. Use this to audit credential access."),
    ("listStorageCredentials", "unityCatalog", "Storage Credential Operations", "List storage credentials. Use this to audit credential discovery."),
    
    # Unity Catalog - Connections
    ("createConnection", "unityCatalog", "Connection Operations", "Create a connection to external data sources like PostgreSQL, MySQL, Snowflake. Use this to track federation setup."),
    ("deleteConnection", "unityCatalog", "Connection Operations", "Delete a connection. Use this to track federation removals."),
    ("updateConnection", "unityCatalog", "Connection Operations", "Update connection properties. Use this to track federation configuration changes."),
    ("getConnection", "unityCatalog", "Connection Operations", "Get connection metadata. Use this to audit federation access."),
    ("listConnections", "unityCatalog", "Connection Operations", "List connections. Use this to audit federation discovery."),
    
    # Unity Catalog - Credentials
    ("generateTemporaryTableCredential", "unityCatalog", "Credential Operations", "Generate temporary credentials for table access. Use this to track credential generation for data access."),
    ("generateTemporaryVolumeCredential", "unityCatalog", "Credential Operations", "Generate temporary credentials for volume access. Use this to track credential generation for file access."),
    ("generateTemporaryPathCredential", "unityCatalog", "Credential Operations", "Generate temporary credentials for path access. Use this to track credential generation for storage paths."),
    
    # Notebook Operations
    ("createNotebook", "notebook", "Notebook Operations", "Create a new notebook. Use this to track notebook creation and authorship."),
    ("deleteNotebook", "notebook", "Notebook Operations", "Delete a notebook. Use this to track notebook removals."),
    ("moveNotebook", "notebook", "Notebook Operations", "Move a notebook to a different location. Use this to track notebook reorganization."),
    ("renameNotebook", "notebook", "Notebook Operations", "Rename a notebook. Use this to track notebook name changes."),
    ("attachNotebook", "notebook", "Notebook Operations", "Attach a notebook to a cluster. Use this to track compute resource binding."),
    ("detachNotebook", "notebook", "Notebook Operations", "Detach a notebook from a cluster. Use this to track compute resource unbinding."),
    ("exportNotebook", "notebook", "Notebook Operations", "Export a notebook to a file. Use this to track data exports and downloads of notebook content."),
    ("importNotebook", "notebook", "Notebook Operations", "Import a notebook from a file. Use this to track notebook imports."),
    
    # Command Execution
    ("commandSubmit", "notebook", "Command Execution", "Submit a command for execution. Use this to track code submissions in notebooks or SQL editors."),
    ("runCommand", "notebook", "Command Execution", "Execute a command. Use this to see actual code executed. Requires verbose audit logs enabled."),
    
    # Folder Operations
    ("createFolder", "workspace", "Folder Operations", "Create a new folder in the workspace. Use this to track workspace organization."),
    ("deleteFolder", "workspace", "Folder Operations", "Delete a folder from the workspace. Use this to track folder removals."),
    ("moveFolder", "workspace", "Folder Operations", "Move a folder to a different location. Use this to track workspace reorganization."),
    
    # Cluster Operations
    ("createCluster", "clusters", "Cluster Operations", "Create a compute cluster. Use this to track cluster provisioning and who created compute resources."),
    ("deleteCluster", "clusters", "Cluster Operations", "Terminate and delete a cluster. Use this to track cluster terminations."),
    ("startCluster", "clusters", "Cluster Operations", "Start a terminated cluster. Use this to track cluster startups."),
    ("restartCluster", "clusters", "Cluster Operations", "Restart a running cluster. Use this to track cluster restarts."),
    ("resizeCluster", "clusters", "Cluster Operations", "Resize a cluster by changing worker count. Use this to track compute scaling."),
    ("editCluster", "clusters", "Cluster Operations", "Edit cluster configuration. Use this to track cluster configuration changes."),
    ("pinCluster", "clusters", "Cluster Operations", "Pin a cluster to prevent auto-termination. Use this to track cluster pinning."),
    ("unpinCluster", "clusters", "Cluster Operations", "Unpin a cluster. Use this to track cluster unpinning."),
    
    # Job Operations
    ("createJob", "jobs", "Job Operations", "Create a new job for scheduled or triggered execution. Use this to track job creation and scheduling setup."),
    ("deleteJob", "jobs", "Job Operations", "Delete a job. Use this to track job removals."),
    ("updateJob", "jobs", "Job Operations", "Update job configuration. Use this to track job configuration changes."),
    ("resetJob", "jobs", "Job Operations", "Reset job to initial state. Use this to track job resets."),
    ("runJob", "jobs", "Job Operations", "Trigger a job run. Use this to track manual or API-triggered job executions."),
    ("cancelRun", "jobs", "Job Operations", "Cancel a running job. Use this to track job cancellations."),
    ("submitRun", "jobs", "Job Operations", "Submit a one-time run. Use this to track one-time job submissions."),
    
    # SQL Warehouse Operations
    ("createWarehouse", "sql", "SQL Warehouse Operations", "Create a SQL warehouse for Databricks SQL. Use this to track warehouse provisioning."),
    ("deleteWarehouse", "sql", "SQL Warehouse Operations", "Delete a SQL warehouse. Use this to track warehouse removals."),
    ("editWarehouse", "sql", "SQL Warehouse Operations", "Edit SQL warehouse configuration. Use this to track warehouse configuration changes."),
    ("startWarehouse", "sql", "SQL Warehouse Operations", "Start a stopped SQL warehouse. Use this to track warehouse startups."),
    ("stopWarehouse", "sql", "SQL Warehouse Operations", "Stop a running SQL warehouse. Use this to track warehouse shutdowns."),
    
    # SQL Query Operations
    ("createQuery", "sql", "SQL Query Operations", "Create a saved SQL query in Databricks SQL. Use this to track query creation."),
    ("deleteQuery", "sql", "SQL Query Operations", "Delete a saved SQL query. Use this to track query removals."),
    ("updateQuery", "sql", "SQL Query Operations", "Update a saved SQL query. Use this to track query modifications."),
    ("executeQuery", "sql", "SQL Query Operations", "Execute a SQL query. Use this to track query executions."),
    
    # Dashboard Operations
    ("createDashboard", "sql", "Dashboard Operations", "Create a dashboard in Databricks SQL. Use this to track dashboard creation."),
    ("deleteDashboard", "sql", "Dashboard Operations", "Delete a dashboard. Use this to track dashboard removals."),
    ("updateDashboard", "sql", "Dashboard Operations", "Update a dashboard. Use this to track dashboard modifications."),
    
    # Alert Operations
    ("createAlert", "sql", "Alert Operations", "Create an alert for monitoring query results. Use this to track alert creation."),
    ("deleteAlert", "sql", "Alert Operations", "Delete an alert. Use this to track alert removals."),
    ("updateAlert", "sql", "Alert Operations", "Update an alert configuration. Use this to track alert modifications."),
    
    # MLflow Operations
    ("createExperiment", "mlflow", "MLflow Operations", "Create an MLflow experiment for tracking ML runs. Use this to track experiment creation."),
    ("deleteExperiment", "mlflow", "MLflow Operations", "Delete an MLflow experiment. Use this to track experiment removals."),
    ("restoreExperiment", "mlflow", "MLflow Operations", "Restore a deleted MLflow experiment. Use this to track experiment restorations."),
    ("createRun", "mlflow", "MLflow Operations", "Create an MLflow run for logging metrics and artifacts. Use this to track ML run creation."),
    ("deleteRun", "mlflow", "MLflow Operations", "Delete an MLflow run. Use this to track ML run removals."),
    ("restoreRun", "mlflow", "MLflow Operations", "Restore a deleted MLflow run. Use this to track ML run restorations."),
    ("logMetric", "mlflow", "MLflow Operations", "Log a metric to an MLflow run. Use this to track metric logging."),
    ("logParam", "mlflow", "MLflow Operations", "Log a parameter to an MLflow run. Use this to track parameter logging."),
    ("logArtifact", "mlflow", "MLflow Operations", "Log an artifact to an MLflow run. Use this to track artifact uploads."),
    
    # Model Registry Operations
    ("createRegisteredModel", "mlflow", "Model Registry Operations", "Register a new model in the model registry. Use this to track model registration."),
    ("deleteRegisteredModel", "mlflow", "Model Registry Operations", "Delete a registered model. Use this to track model removals."),
    ("renameRegisteredModel", "mlflow", "Model Registry Operations", "Rename a registered model. Use this to track model renames."),
    ("createModelVersion", "mlflow", "Model Registry Operations", "Create a new version of a registered model. Use this to track model version creation."),
    ("deleteModelVersion", "mlflow", "Model Registry Operations", "Delete a model version. Use this to track model version removals."),
    ("transitionModelVersionStage", "mlflow", "Model Registry Operations", "Transition model version to a new stage (Staging, Production, Archived). Use this to track model deployments."),
    
    # Model Serving Operations
    ("createServingEndpoint", "serving", "Model Serving Operations", "Create a model serving endpoint. Use this to track endpoint creation for real-time inference."),
    ("deleteServingEndpoint", "serving", "Model Serving Operations", "Delete a model serving endpoint. Use this to track endpoint removals."),
    ("updateServingEndpoint", "serving", "Model Serving Operations", "Update a model serving endpoint configuration. Use this to track endpoint configuration changes."),
    
    # Secret Operations
    ("createSecretScope", "secrets", "Secret Operations", "Create a secret scope for storing sensitive data. Use this to track secret scope creation."),
    ("deleteSecretScope", "secrets", "Secret Operations", "Delete a secret scope. Use this to track secret scope removals."),
    ("putSecret", "secrets", "Secret Operations", "Store a secret in a scope. Use this to track secret creation and updates."),
    ("deleteSecret", "secrets", "Secret Operations", "Delete a secret from a scope. Use this to track secret removals."),
    ("getSecret", "secrets", "Secret Operations", "Retrieve a secret value. Use this to audit secret access."),
    ("listSecrets", "secrets", "Secret Operations", "List secrets in a scope. Use this to audit secret discovery."),
    
    # Token Operations
    ("createToken", "tokens", "Token Operations", "Create a personal access token (PAT). Use this to track API token creation."),
    ("revokeToken", "tokens", "Token Operations", "Revoke a personal access token. Use this to track API token revocations."),
    ("listTokens", "tokens", "Token Operations", "List personal access tokens. Use this to audit token inventory."),
    
    # Authentication Events
    ("login", "accounts", "Authentication", "User login to workspace. Use this to track user authentication and login attempts."),
    ("logout", "accounts", "Authentication", "User logout from workspace. Use this to track user session endings."),
    ("tokenLogin", "accounts", "Authentication", "Login using a token. Use this to track API authentication."),
    ("oidcTokenAuthorization", "accounts", "Authentication", "OIDC token authorization. Use this to track SSO authentication."),
    ("samlLogin", "accounts", "Authentication", "SAML-based login. Use this to track SSO authentication via SAML."),
    
    # User Management
    ("createUser", "accounts", "User Management", "Add a new user to the workspace or account. Use this to track user provisioning."),
    ("deleteUser", "accounts", "User Management", "Remove a user from the workspace or account. Use this to track user deprovisioning."),
    ("updateUser", "accounts", "User Management", "Update user properties. Use this to track user attribute changes."),
    ("setAdmin", "accounts", "User Management", "Set user as admin. Use this to track admin privilege grants."),
    ("removeAdmin", "accounts", "User Management", "Remove admin privileges from user. Use this to track admin privilege revocations."),
    
    # Group Management
    ("createGroup", "accounts", "Group Management", "Create a new group. Use this to track group creation for access management."),
    ("deleteGroup", "accounts", "Group Management", "Delete a group. Use this to track group removals."),
    ("updateGroup", "accounts", "Group Management", "Update group properties. Use this to track group attribute changes."),
    ("addPrincipalToGroup", "accounts", "Group Management", "Add a user or service principal to a group. Use this to track group membership additions."),
    ("removePrincipalFromGroup", "accounts", "Group Management", "Remove a user or service principal from a group. Use this to track group membership removals."),
    
    # Service Principal Operations
    ("createServicePrincipal", "accounts", "Service Principal Operations", "Create a service principal for automated access. Use this to track service account creation."),
    ("deleteServicePrincipal", "accounts", "Service Principal Operations", "Delete a service principal. Use this to track service account removals."),
    ("updateServicePrincipal", "accounts", "Service Principal Operations", "Update service principal properties. Use this to track service account changes."),
    
    # Databricks Apps
    ("createApp", "apps", "Databricks Apps", "Create a Databricks app. Use this to track app deployments."),
    ("deleteApp", "apps", "Databricks Apps", "Delete a Databricks app. Use this to track app removals."),
    ("updateApp", "apps", "Databricks Apps", "Update a Databricks app. Use this to track app configuration changes."),
    ("deployApp", "apps", "Databricks Apps", "Deploy a Databricks app version. Use this to track app deployments."),
    ("changeAppsAcl", "apps", "Databricks Apps", "Modify app access control list. Use this to audit app sharing and access changes."),
    ("workspaceInHouseOAuthClientAuthentication", "apps", "Databricks Apps", "OAuth client authentication for workspace apps. Use this to audit app user logins."),
    ("mintOAuthToken", "apps", "Databricks Apps", "Generate OAuth token for app access. Use this to track OAuth token generation."),
    ("mintOAuthAuthorizationCode", "apps", "Databricks Apps", "Generate OAuth authorization code. Use this to track OAuth authorization flow."),
    
    # Git/Repos Operations
    ("createRepo", "repos", "Git Operations", "Create a Git repo connection. Use this to track repo setup."),
    ("deleteRepo", "repos", "Git Operations", "Delete a Git repo connection. Use this to track repo removals."),
    ("updateRepo", "repos", "Git Operations", "Update repo settings or pull changes. Use this to track repo updates."),
    ("commitAndPush", "repos", "Git Operations", "Commit and push changes to remote. Use this to track code pushes."),
    ("pull", "repos", "Git Operations", "Pull changes from remote. Use this to track code pulls."),
    
    # Delta Live Tables
    ("createPipeline", "pipelines", "Delta Live Tables", "Create a Delta Live Tables pipeline. Use this to track DLT pipeline creation."),
    ("deletePipeline", "pipelines", "Delta Live Tables", "Delete a Delta Live Tables pipeline. Use this to track DLT pipeline removals."),
    ("updatePipeline", "pipelines", "Delta Live Tables", "Update a Delta Live Tables pipeline. Use this to track DLT pipeline configuration changes."),
    ("startPipeline", "pipelines", "Delta Live Tables", "Start a Delta Live Tables pipeline run. Use this to track DLT pipeline executions."),
    ("stopPipeline", "pipelines", "Delta Live Tables", "Stop a running Delta Live Tables pipeline. Use this to track DLT pipeline stops."),
    
    # Vector Search
    ("createVectorSearchEndpoint", "vectorSearch", "Vector Search Operations", "Create a Vector Search endpoint. Use this to track vector search infrastructure creation."),
    ("deleteVectorSearchEndpoint", "vectorSearch", "Vector Search Operations", "Delete a Vector Search endpoint. Use this to track vector search infrastructure removals."),
    ("createVectorSearchIndex", "vectorSearch", "Vector Search Operations", "Create a Vector Search index. Use this to track vector index creation."),
    ("deleteVectorSearchIndex", "vectorSearch", "Vector Search Operations", "Delete a Vector Search index. Use this to track vector index removals."),
    ("queryVectorSearchIndex", "vectorSearch", "Vector Search Operations", "Query a Vector Search index. Use this to audit vector search queries."),
    
    # Delta Sharing
    ("createShare", "shares", "Delta Sharing", "Create a share for sharing data externally. Use this to track share creation."),
    ("deleteShare", "shares", "Delta Sharing", "Delete a share. Use this to track share removals."),
    ("updateShare", "shares", "Delta Sharing", "Update share configuration. Use this to track share changes."),
    ("createRecipient", "shares", "Delta Sharing", "Create a sharing recipient. Use this to track recipient setup."),
    ("deleteRecipient", "shares", "Delta Sharing", "Delete a sharing recipient. Use this to track recipient removals."),
    ("updateRecipient", "shares", "Delta Sharing", "Update recipient configuration. Use this to track recipient changes."),
    ("createProvider", "shares", "Delta Sharing", "Create a sharing provider. Use this to track provider setup."),
    ("deleteProvider", "shares", "Delta Sharing", "Delete a sharing provider. Use this to track provider removals."),
    
    # Data Export/Download
    ("downloadResults", "sql", "Data Export", "Download query results. Use this to audit data downloads and exports."),
    ("downloadNotebook", "workspace", "Data Export", "Download a notebook file. Use this to audit notebook exports."),
    ("downloadFile", "workspace", "Data Export", "Download a file from workspace. Use this to audit file downloads."),
    
    # DBFS Operations
    ("dbfsCreate", "dbfs", "DBFS Operations", "Create a file in DBFS. Use this to track file creation in Databricks File System."),
    ("dbfsDelete", "dbfs", "DBFS Operations", "Delete a file from DBFS. Use this to track file deletions in Databricks File System."),
    ("dbfsRead", "dbfs", "DBFS Operations", "Read a file from DBFS. Use this to audit file reads in Databricks File System."),
    ("dbfsList", "dbfs", "DBFS Operations", "List files in DBFS. Use this to audit file discovery in Databricks File System."),
    
    # Workspace Configuration
    ("workspaceConfEdit", "workspace", "Workspace Configuration", "Edit workspace configuration settings. Use this to track workspace setting changes."),
    ("workspaceConfDelete", "workspace", "Workspace Configuration", "Delete workspace configuration. Use this to track workspace setting removals."),
    
    # IP Access Lists
    ("createIpAccessList", "accounts", "IP Access Lists", "Create an IP access list for workspace security. Use this to track IP allowlist/blocklist creation."),
    ("deleteIpAccessList", "accounts", "IP Access Lists", "Delete an IP access list. Use this to track IP allowlist/blocklist removals."),
    ("updateIpAccessList", "accounts", "IP Access Lists", "Update an IP access list. Use this to track IP allowlist/blocklist changes."),
    
    # Instance Pools
    ("createInstancePool", "clusters", "Instance Pool Operations", "Create an instance pool for faster cluster startup. Use this to track pool creation."),
    ("deleteInstancePool", "clusters", "Instance Pool Operations", "Delete an instance pool. Use this to track pool removals."),
    ("editInstancePool", "clusters", "Instance Pool Operations", "Edit instance pool configuration. Use this to track pool configuration changes."),
    
    # Cluster Policies
    ("createClusterPolicy", "clusters", "Cluster Policy Operations", "Create a cluster policy for governance. Use this to track policy creation."),
    ("deleteClusterPolicy", "clusters", "Cluster Policy Operations", "Delete a cluster policy. Use this to track policy removals."),
    ("editClusterPolicy", "clusters", "Cluster Policy Operations", "Edit cluster policy configuration. Use this to track policy changes."),
]

print(f"Total audit actions defined: {len(AUDIT_ACTIONS)}")

In [ ]:
# Create table
if FRESH_START:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
    spark.sql(f"DROP VIEW IF EXISTS {FULL_TABLE_NAME}")
    print("Fresh start - dropped existing table")

empty_df = spark.createDataFrame([], schema)
empty_df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(FULL_TABLE_NAME)

spark.sql(f"""
    COMMENT ON TABLE {FULL_TABLE_NAME} IS 
    'Azure Databricks audit log action_names with descriptions. Query with Vector Search to find relevant actions for auditing intent.'
""")
print(f"Created table {FULL_TABLE_NAME} with CDF enabled")

In [ ]:
# Build records with embedding text
records = []
for action_name, service_name, category, description in AUDIT_ACTIONS:
    # Embedding text combines category and description for better semantic matching
    embedding_text = f"{category}: {description}"
    records.append((action_name, service_name, category, description, embedding_text))

# Write all records
df = spark.createDataFrame(records, schema)
df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)

print(f"Loaded {len(records)} audit actions")

In [ ]:
# Verify
count = spark.sql(f"SELECT COUNT(*) FROM {FULL_TABLE_NAME}").collect()[0][0]
print(f"Total records in table: {count}")

# Show distribution by category
print("\nRecords by category:")
display(spark.sql(f"""
    SELECT category, COUNT(*) as count 
    FROM {FULL_TABLE_NAME} 
    GROUP BY category 
    ORDER BY count DESC
"""))

In [ ]:
# Preview records
display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} ORDER BY category, action_name"))